In [2]:
# =========================================================
# Notebook : Création du Bronze Layer pour Telco Churn
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException

# ------------------------
# Configuration MinIO
# ------------------------
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "telco-churn"
MINIO_ENDPOINT = "minio1:9000"

# ------------------------
# Initialisation Spark
# ------------------------
spark = SparkSession.builder \
    .appName("TelcoChurn_Bronze") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# ------------------------
# Lecture des CSV bruts
# ------------------------
input_path = f"s3a://{MINIO_BUCKET}/raw/*.csv"

try:
    df_bronze = spark.read.csv(
        input_path,
        header=True,
        inferSchema=True,
        mode="DROPMALFORMED"
    )
    
    print(f"✅ CSV bruts chargés : {df_bronze.count()} lignes, {len(df_bronze.columns)} colonnes")
    df_bronze.show(10, truncate=False)

    # ------------------------
    # Sauvegarde en Bronze (Delta Lake)
    # ------------------------
    bronze_path = f"s3a://{MINIO_BUCKET}/bronze/telco_churn"
    
    df_bronze.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(bronze_path)
    
    print(f"✅ Bronze Layer créé avec succès : {bronze_path}")

except AnalysisException as e:
    print(f"❌ Spark AnalysisException : {e}")
except Exception as e:
    print(f"❌ Erreur : {e}")


25/12/17 18:56:22 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


✅ CSV bruts chargés : 7041 lignes, 21 colonnes
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+-------------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines   |InternetService|OnlineSecurity     |OnlineBackup       |DeviceProtection   |TechSupport        |StreamingTV        |StreamingMovies    |Contract      |PaperlessBilling|PaymentMethod            |MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+-------------------------+--------------+------------+-----+

25/12/17 18:59:08 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
25/12/17 18:59:08 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:981)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:165)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:263)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:170)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.proce